In [9]:
# Task 1.3
import sqlite3

conn = sqlite3.connect('./quickbuy.db')
cursor = conn.cursor()

cursor.executescript('''
DROP TABLE customers;
DROP TABLE products;
DROP TABLE orders;
''')

cursor.executescript('''
CREATE TABLE IF NOT EXISTS customers(
    customer_id TEXT PRIMARY KEY,
    customer_name TEXT NOT NULL,
    email TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS products(
    product_id TEXT PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS orders(
    customer_id TEXT NOT NULL,
    product_id TEXT NOT NULL,
    order_date TEXT NOT NULL,
    quantity INT NOT NULL,

    PRIMARY KEY (customer_id, product_id, order_date),

    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
''')

conn.commit()
conn.close()

In [11]:
# Task 3.2
import csv

conn = sqlite3.connect('./quickbuy.db')
cursor = conn.cursor()

with open('./RESOURCES/customers.txt') as f:
    data = [i for i in csv.reader(f)]
    for info in data:
        cursor.execute('INSERT INTO customers(customer_id, customer_name, email) VALUES (?, ?, ?)', info)

with open('./RESOURCES/products.txt') as f:
    data = [i for i in csv.reader(f)]
    for info in data:
        cursor.execute('INSERT INTO products(product_id, product_name, category) VALUES (?, ?, ?)', info)

with open('./RESOURCES/orders.txt') as f:
    data = [i for i in csv.reader(f)]
    for info in data:
        cursor.execute('INSERT INTO orders(customer_id, product_id, order_date, quantity) VALUES (?, ?, ?, ?)', info)

conn.commit()
conn.close()

In [19]:
# Task 3.3
conn = sqlite3.connect('./quickbuy.db')
cursor = conn.cursor()

customers = {}
cursor.execute('SELECT * FROM orders')
orders = cursor.fetchall()
for info in orders:
    if customers.get(info[0]):
        customers[info[0]].append(info[1:])
    else:
        customers[info[0]] = [info[1:]]
        
for k, v in customers.items():
    print(f'{k}: {len(v)}')

conn.close()

C1: 3
C2: 3
C3: 3
C4: 3
C5: 3
C6: 3
C7: 3
C8: 3
C9: 3
C10: 3


In [29]:
# Task 3.4
conn = sqlite3.connect('./quickbuy.db')
cursor = conn.cursor()

cursor.execute('SELECT p.category, o.order_date, o.quantity FROM orders as o, products as p WHERE p.product_id = o.product_id')
orders = cursor.fetchall()
#print(orders)

products = {}
for info in orders:
    if int(info[1].split('-')[2]) != 23: continue
    
    if products.get(info[0]):
        products[info[0]] += int(info[2])
    else:
        products[info[0]] = int(info[2])

print(products)
conn.close()

{'Electronics': 1}


In [33]:
# Task 3.5
import json
from pymongo import MongoClient

client = MongoClient('localhost', 27017)
db = client['quickbuy']
coll = db['reviews']

with open('./RESOURCES/reviews.json') as f:
    data = json.load(f)
    #print(data)

coll.insert_many(data)

[{'product_id': 'P5', 'product_name': 'Ultra HD Smart TV', 'customer_id': 'C1', 'rating': 5, 'review_text': 'Amazing picture quality and smart features! The colors are vibrant and the interface is user-friendly.', 'review_date': '2024-08-28'}, {'product_id': 'P3', 'product_name': 'Ergonomic Office Chair', 'customer_id': 'C2', 'rating': 4, 'review_text': 'Very comfortable for long working hours. The lumbar support is great, but I wish the armrests were a bit more adjustable.', 'review_date': '2024-08-27'}, {'product_id': 'P2', 'product_name': 'Smartphone XYZ', 'customer_id': 'C3', 'rating': 3, 'review_text': "Decent phone with good camera, but the battery life is less than I expected. It's okay for the price, but not outstanding.", 'review_date': '2024-08-26'}, {'product_id': 'P4', 'product_name': 'UltraBook Pro', 'customer_id': 'C4', 'rating': 5, 'review_date': '2024-08-25'}, {'product_id': 'P1', 'product_name': 'Noise-Cancelling Headphones', 'customer_id': 'C5', 'rating': 2, 'review_d

In [45]:
# Task 3.6
client = MongoClient('localhost', 27017)
db = client['quickbuy']
coll = db['reviews']

data = coll.find({
    'rating': {'$gt': 3},
    'review_text': {'$ne': None}
})

print([(i['rating'], i['review_text']) for i in data])

[(5, 'Amazing picture quality and smart features! The colors are vibrant and the interface is user-friendly.'), (4, 'Very comfortable for long working hours. The lumbar support is great, but I wish the armrests were a bit more adjustable.')]


In [48]:
# Task 3.7
client = MongoClient('localhost', 27017)
db = client['quickbuy']
coll = db['reviews']

data = list(coll.find({
    'product_id': 'P5'
}))

print(f'{sum([int(i['rating']) for i in data])/len(data):.1f}')

4.5
